In [1]:
import numpy as np
import pandas as pd
import os
import sys
import glob
import tqdm
import re
from typing import List

sys.path.append("..")
from src import text_extraction, create_sentence_nace_code_similarities, analysis_functions
import test_base
from sentence_splitter import split_text_into_sentences

/Users/hendrikweichel/miniconda3/envs/nace_project/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Test retrieving the similarities for chunks in a pdf to the NACE Code

**Function:** pdf-> (chunk x code -> [-1,1])

**Parameters:** 

- pdf_path
- way of chunking the text (e.g. sentences, sliding window, or paragraphs)
- way of preprocessing (most is fixed for all reports)
    - similarity threshold of relevant chunks
    - length of irrelevant chunks

**Store analytics for each datapoint:**

- mean score for each class given a threshold

In [2]:
# Parameters: 

threshold_min_chunk_len = 100
threshold_min_paragraph_len = 0
cos_threshold = 0.4
sentence_length = 6

In [3]:
dataset_path = "../data/german_annual_reports"
dataset_path = "../data/PDF_stoxx600"
dataset_path_texts = "../data/TEXT_stoxx600_docling"

dataset_path = "../data/PDF_stoxx_extended"
dataset_path_texts = "../data/TEXT_stoxx600_extended_docling"

In [4]:
nace_classes = pd.read_excel(os.path.join(dataset_path, "STOXX600_extended.xlsx"))
nace_classes.head()

,Name,Symbol,FactSet ID,Revenue - 2022 (in EUR),Revenue - 2023 (in EUR),Revenue - 2024 (in EUR),NACE,NACE_letter,Report
0,SalMar ASA,SALM-NO,SALM-NO,1984.965581,2458.824022,2271.611663,3.21,A,salmar-annual-report-2022.pdf
1,Bakkafrost P/F,BAKKA-NO,BAKKA-NO,929.503079,937.862170,973.053513,3.21,A,Bakkafrost PF2.pdf
2,Antofagasta plc,ANTO-GB,ANTO-GB,5577.681426,5849.975673,6113.946983,7.29,B,Antofagasta plc1.pdf
3,Anglo American plc,AAL-GB,AAL-GB,33423.271144,28355.894415,25288.188492,7.29,B,Anglo American plc1.pdf
4,TotalEnergies SE,TTE-FR,TTE-FR,250538.948328,202517.658053,180837.266896,6.10,B,Totalenergies EP Gabon1.pdf


In [5]:
report_to_nace_class = nace_classes.dropna(subset=["Report"]).set_index('Report').to_dict()["NACE"]
report_to_nace_class

{'salmar-annual-report-2022.pdf': 3.21,
 'Bakkafrost PF2.pdf': 3.21,
 'Antofagasta plc1.pdf': 7.29,
 'Anglo American plc1.pdf': 7.29,
 'Totalenergies EP Gabon1.pdf': 6.1,
 'subsea_2022-Annual-Report.pdf': 9.1,
 'shell-annual-report-2022.pdf': 6.1,
 'LANXESS AG1.pdf': 20.59,
 'Gerresheimer AG1.pdf': 22.22,
 'Verallia SAS3.pdf': 23.14,
 'BELIMO Holding AG1.pdf': 28.12,
 'Diageo PLC1.pdf': 11.01,
 'BAE Systems plc1.pdf': 30.3,
 'British American Tobacco p.l.c.1.pdf': 12.0,
 'Games Workshop Group PLC1.pdf': 32.4,
 'Greggs plc1.pdf': 10.71,
 'Halma plc1.pdf': 26.51,
 'Imperial Brands PLC3.pdf': 12.0,
 'IMI plc1.pdf': 28.99,
 'Howden Joinery Group PLC2.pdf': 27.51,
 'Associated British Foods plc1.pdf': 10.89,
 'Signify NV3.pdf': 27.4,
 'Smiths Group PLC1.pdf': 28.15,
 'Tate & Lyle PLC1.pdf': 10.89,
 'Smith & Nephew plc1.pdf': 32.5,
 'GSK PLC1.pdf': 21.2,
 'AstraZeneca PLC1.pdf': 21.2,
 'Burberry Group plc2.pdf': 14.19,
 'Airbus SE1.pdf': 30.3,
 "L'Oreal S.A.1.pdf": 20.42,
 'Dassault Aviation

In [6]:
report_to_nace_class = {report[0][:-4] + ".txt": report[1] for report in report_to_nace_class.items()}
report_to_nace_class

{'salmar-annual-report-2022.txt': 3.21,
 'Bakkafrost PF2.txt': 3.21,
 'Antofagasta plc1.txt': 7.29,
 'Anglo American plc1.txt': 7.29,
 'Totalenergies EP Gabon1.txt': 6.1,
 'subsea_2022-Annual-Report.txt': 9.1,
 'shell-annual-report-2022.txt': 6.1,
 'LANXESS AG1.txt': 20.59,
 'Gerresheimer AG1.txt': 22.22,
 'Verallia SAS3.txt': 23.14,
 'BELIMO Holding AG1.txt': 28.12,
 'Diageo PLC1.txt': 11.01,
 'BAE Systems plc1.txt': 30.3,
 'British American Tobacco p.l.c.1.txt': 12.0,
 'Games Workshop Group PLC1.txt': 32.4,
 'Greggs plc1.txt': 10.71,
 'Halma plc1.txt': 26.51,
 'Imperial Brands PLC3.txt': 12.0,
 'IMI plc1.txt': 28.99,
 'Howden Joinery Group PLC2.txt': 27.51,
 'Associated British Foods plc1.txt': 10.89,
 'Signify NV3.txt': 27.4,
 'Smiths Group PLC1.txt': 28.15,
 'Tate & Lyle PLC1.txt': 10.89,
 'Smith & Nephew plc1.txt': 32.5,
 'GSK PLC1.txt': 21.2,
 'AstraZeneca PLC1.txt': 21.2,
 'Burberry Group plc2.txt': 14.19,
 'Airbus SE1.txt': 30.3,
 "L'Oreal S.A.1.txt": 20.42,
 'Dassault Aviation

In [7]:
reports_path = glob.glob(os.path.join(dataset_path_texts, "*.txt"))
reports_path

['../data/TEXT_stoxx600_extended_docling/Hannover Rueck SE1.txt',
 '../data/TEXT_stoxx600_extended_docling/Interpump Group S.p.A.1.txt',
 '../data/TEXT_stoxx600_extended_docling/Intertek Group PLC1.txt',
 '../data/TEXT_stoxx600_extended_docling/Advanced Health Limited2.txt',
 '../data/TEXT_stoxx600_extended_docling/Anheuser-Busch InBev SANV3.txt',
 '../data/TEXT_stoxx600_extended_docling/Scout24 SE3.txt',
 '../data/TEXT_stoxx600_extended_docling/Bridgepoint Group Plc1.txt',
 '../data/TEXT_stoxx600_extended_docling/ad pepper media International N.V.1.txt',
 '../data/TEXT_stoxx600_extended_docling/Financiere de Tubize SA2.txt',
 '../data/TEXT_stoxx600_extended_docling/Severn Trent Plc1.txt',
 '../data/TEXT_stoxx600_extended_docling/Bakkafrost PF2.txt',
 '../data/TEXT_stoxx600_extended_docling/Ferrari NV2.txt',
 '../data/TEXT_stoxx600_extended_docling/Ardent Leisure Group Ltd1.txt',
 '../data/TEXT_stoxx600_extended_docling/Acadia Healthcare Company, Inc.1.txt',
 '../data/TEXT_stoxx600_ext

In [8]:
def get_tables(lines: list): 
    tables = []
    current_table = []

    for line in lines:
        if line.strip().startswith("|"):  # line belongs to a table
            current_table.append(line.strip())
        else:
            if current_table:  # table ended
                tables.append("\n".join(current_table))
                current_table = []

    # catch last table if file ends without empty lines
    if current_table:
        tables.append("\n".join(current_table))

    return tables

def preprocess_report(pdf_path: str) -> List[str]:

    with open(pdf_path, "r") as f: 
        text = f.read()
    
    lines = text.split("\n")

    tables = get_tables(lines)

    # drop if condidtion is True
    conditions = [
        # filter images
        lambda line: line == '<!-- image -->',
        
        #filter tables 
        lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False, 

        # filter headers
        lambda line: line.strip()[0] == "#" if len(line) > 0 else True,

        # filter sentences
        lambda line: "." not in line,
        
        # more than 50% is numbers
        lambda line: sum(ch.isalpha() for ch in line) / len(line) < 0.5,

        # minimum 3 words 
        lambda line: len(re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", '', line).strip().split(" ")) < 3,

        # Minimum 2 Sentences
        #lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

    ]
    accepted_lines = [line for line in lines if not any(condition(line) for condition in conditions)]
    accepted_lines += tables

    chunks = []

    for line in accepted_lines: 
        sentences = split_text_into_sentences(line, language='en')
        sentences = [sentence.strip() for sentence in sentences]
        sentences = [sentence for sentence in sentences if sentence != ""]
        new_chunks = [(" ".join(sentences[i:i+sentence_length])).strip() for i in range(0, len(sentences), 3)]

        chunks += new_chunks

    # if there is only one sentence in the last chunk, balance the two last chunks
    if len(split_text_into_sentences(chunks[-1], language = "en")) == 1: 
        last_two_chunks = chunks[-2] + " " + chunks[-1]
        chunks[-2] = last_two_chunks[0:(len(last_two_chunks) + 1) // 2]
        chunks[-1] = last_two_chunks[(len(last_two_chunks) + 1) // 2: (len(last_two_chunks)) - (len(last_two_chunks) + 1) // 2]

    chunks = [re.sub(r'\b\d+\.\d+\b', '', chunk) for chunk in chunks]
    chunks = [re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", '', chunk) for chunk in chunks]
    chunks = [re.sub(r"\s+", " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'\.{2,}', " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'^\d+\.\s*', " ", chunk) for chunk in chunks]
    chunks = [chunk.lower() for chunk in chunks]
    chunks = [chunk.strip() for chunk in chunks]

    return chunks

In [9]:
for i in range(1, 2): 
    nace_level = i

    result_path = f"../results/paragraph_and_sentence_len_{sentence_length}_min_chunk_len_{threshold_min_chunk_len}_cos_thresh_{cos_threshold}_nace_level_{nace_level}_stoxx_extended"

    res = test_base.test_similarities(reports_path, preprocess_report, threshold_min_chunk_len, cos_threshold, report_to_nace_class, result_path, level=i)

  0%|          | 0/37 [00:00<?, ?it/s]

Report:  ../data/TEXT_stoxx600_extended_docling/Eurofins Scientific SE2.txt
Number of Chunks:  31


  3%|▎         | 1/37 [00:18<10:49, 18.04s/it]

Report:  ../data/TEXT_stoxx600_extended_docling/TAG Immobilien AG3.txt
Number of Chunks:  1099


  5%|▌         | 2/37 [00:27<07:40, 13.15s/it]

...

 95%|█████████▍| 35/37 [80:23:06<07:14, 217.02s/it]

Report:  ../data/TEXT_stoxx600_extended_docling/Phoenix Group Holdings plc1.txt
Number of Chunks:  2369


 97%|█████████▋| 36/37 [80:23:25<02:37, 157.59s/it]

Report:  ../data/TEXT_stoxx600_extended_docling/Bellway p.l.c.1.txt
Number of Chunks:  1372


100%|██████████| 37/37 [80:23:35<00:00, 7822.04s/it]


In [10]:
nace_level = 1
for i in [3,4,5,6,7]: 

    sentence_length = i

    def preprocess_report(pdf_path: str) -> List[str]:

        with open(pdf_path, "r") as f: 
            text = f.read()
        
        lines = text.split("\n")

        # drop if condidtion is True
        conditions = [
            # filter images
            lambda line: line == '<!-- image -->',
            
            #filter tables 
            lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False, 

            # filter headers
            lambda line: line.strip()[0] == "#" if len(line) > 0 else True,

            # filter sentences
            lambda line: "." not in line,
            
            # more than 50% is numbers
            lambda line: sum(ch.isalpha() for ch in line) / len(line) < 0.5,

            # minimum 3 words 
            lambda line: len(re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", '', line).strip().split(" ")) < 3,

            # Minimum 2 Sentences
            #lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

        ]
        accepted_lines = [line for line in lines if not any(condition(line) for condition in conditions)]

        chunks = []

        for line in accepted_lines: 
            sentences = split_text_into_sentences(line, language='en')
            sentences = [sentence.strip() for sentence in sentences]
            sentences = [sentence for sentence in sentences if sentence != ""]
            new_chunks = [(" ".join(sentences[i:i+sentence_length])).strip() for i in range(0, len(sentences), 3)]

            chunks += new_chunks

        # if there is only one sentence in the last chunk, balance the two last chunks
        if len(split_text_into_sentences(chunks[-1], language = "en")) == 1: 
            last_two_chunks = chunks[-2] + " " + chunks[-1]
            chunks[-2] = last_two_chunks[0:(len(last_two_chunks) + 1) // 2]
            chunks[-1] = last_two_chunks[(len(last_two_chunks) + 1) // 2: (len(last_two_chunks)) - (len(last_two_chunks) + 1) // 2]

        chunks = [re.sub(r'\b\d+\.\d+\b', '', chunk) for chunk in chunks]
        chunks = [re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", '', chunk) for chunk in chunks]
        chunks = [re.sub(r"\s+", " ", chunk) for chunk in chunks]
        chunks = [re.sub(r'\.{2,}', " ", chunk) for chunk in chunks]
        chunks = [re.sub(r'^\d+\.\s*', " ", chunk) for chunk in chunks]
        chunks = [chunk.lower() for chunk in chunks]
        chunks = [chunk.strip() for chunk in chunks]

        return chunks
    
    result_path = f"../results/paragraph_and_sentence_len_{sentence_length}_min_chunk_len_{threshold_min_chunk_len}_cos_thresh_{cos_threshold}_nace_level_{nace_level}_stoxx"

    res = test_base.test_similarities(reports_path, preprocess_report, threshold_min_chunk_len, cos_threshold, report_to_nace_class, result_path, level=i)

  0%|          | 0/37 [00:00<?, ?it/s]

Report:  ../data/TEXT_stoxx600_extended_docling/Eurofins Scientific SE2.txt
Number of Chunks:  30


  3%|▎         | 1/37 [00:10<06:20, 10.58s/it]

Report:  ../data/TEXT_stoxx600_extended_docling/TAG Immobilien AG3.txt
Number of Chunks:  1098


  5%|▌         | 2/37 [00:48<15:33, 26.67s/it]

...

 95%|█████████▍| 35/37 [07:20<00:15,  7.92s/it]

Report:  ../data/TEXT_stoxx600_extended_docling/Phoenix Group Holdings plc1.txt
Number of Chunks:  2369


 97%|█████████▋| 36/37 [07:50<00:14, 14.61s/it]

Report:  ../data/TEXT_stoxx600_extended_docling/Bellway p.l.c.1.txt
Number of Chunks:  1372


100%|██████████| 37/37 [07:58<00:00, 12.94s/it]
